# 🗄️ 05. SQL-Based Data Analysis

**Objective:** Use SQL interface (DuckDB) to query pandas dataframes for transactional insights.

---


## 1. Import Libraries & Setup DuckDB

In [ ]:
import pandas as pd
import numpy as np
import duckdb
import os
print("DuckDB environment configured.")

## 2. Register Pandas DataFrames with DuckDB
DuckDB allows us to run standard SQL commands directly against Python variables (pandas DataFrames).

In [ ]:
# Register variables
data_dir = "../../data/raw"
orders_df = pd.read_csv(os.path.join(data_dir, "orders.csv"))
customers_df = pd.read_csv(os.path.join(data_dir, "customers.csv"))
con = duckdb.connect(database=':memory:')
con.register('orders', orders_df)
con.register('customers', customers_df)
print("Registered DataFrames as SQL tables.")

## 3. SQL Query: Top 10 Customers by Order Count

In [ ]:
query_top_cust = """
SELECT customer_id, count(order_id) as total_orders
FROM orders
GROUP BY customer_id
ORDER BY total_orders DESC
LIMIT 10
"""
print(con.execute(query_top_cust).df())

## 4. SQL Query: Monthly Growth using Window Functions

In [ ]:
query_growth = """
SELECT 
  strftime('%Y-%m', CAST(order_purchase_timestamp AS TIMESTAMP)) as month,
  count(order_id) as current_orders,
  lag(count(order_id)) over (order by strftime('%Y-%m', CAST(order_purchase_timestamp AS TIMESTAMP))) as prev_orders
FROM orders
GROUP BY month
ORDER BY month
"""
print(con.execute(query_growth).df().head())